# Problem 2 - Transformer from scratch

Run this notebook from top to bottom. Full mode writes samples, figures, and CSV evidence.

In [1]:
import os
REPO = "/content/BITS_Assignments"
if not os.path.exists(REPO):
    !git clone -q https://github.com/GreedyM0nk/BITS_Assignments.git {REPO}
%cd {REPO}
!git checkout -q 4943872
!git log --oneline -1
%cd "/content/BITS_Assignments/1st_Semester/DNN/2nd Assignment/2024SAMPLE_DNN_Assignment/problem2"

/content/BITS_Assignments
4943872 (HEAD) Strengthen notebook and report evidence workflow
/content/BITS_Assignments/1st_Semester/DNN/2nd Assignment/2024SAMPLE_DNN_Assignment/problem2


In [2]:
from pathlib import Path
import sys
import torch
FAST_MODE = False
ROOT = Path.cwd().parent if Path.cwd().name == 'problem2' else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'problem2'))
from problem2 import (
    hand_check_attention, plot_hand_check, run_har_transformer,
    run_transformer_ablations, hand_attention_table,
    run_mask_and_context_ablation, train_language_model,
    generate_samples, plot_attention_heatmaps,
    CausalTransformerLM,
)
import time
out, weights = hand_check_attention()
print(out.shape, weights.shape)
q_hand, k_hand, v_hand, scores, hand_weights, hand_output = hand_attention_table()
print('Q=', q_hand, 'K=', k_hand, 'V=', v_hand)
print('hand scores=', scores, 'hand weights=', hand_weights, 'hand output=', hand_output)
plot_hand_check(ROOT / 'problem2' / 'figures' / 'p2_hand_attention.png')
assert torch.allclose(weights.sum(-1), torch.ones(1, 3), atol=1e-6)
assert torch.allclose(hand_output, hand_weights @ v_hand, atol=1e-6)
from src.models import encoder_block_parameter_breakdown
breakdown = encoder_block_parameter_breakdown(128)
one_block = CausalTransformerLM(65, 128, 4, 1, 128)
print('Encoder weight derivation:', breakdown['formula'], breakdown['weights_only'])
print('One-block parameters:', sum(p.numel() for p in one_block.parameters()))
assert breakdown['weights_only'] == 196608


torch.Size([1, 3, 2]) torch.Size([1, 3, 3])
Q= tensor([[1., 0.],
        [0., 1.],
        [1., 1.]]) K= tensor([[1., 1.],
        [1., 0.],
        [0., 1.]]) V= tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
hand scores= tensor([[0.7071, 0.7071, 0.0000],
        [0.7071, 0.0000, 0.7071],
        [1.4142, 0.7071, 0.7071]]) hand weights= tensor([[0.4011, 0.4011, 0.1978],
        [0.4011, 0.1978, 0.4011],
        [0.5035, 0.2483, 0.2483]]) hand output= tensor([[2.5933, 3.5933],
        [3.0000, 4.0000],
        [2.4895, 3.4895]])
Encoder weight derivation: 4d^2 + 2*d*d_ff 196608
One-block parameters: 215233


## Full HAR Transformer positional ablation

Runs all three seeds for no, sinusoidal, and learned positional information.

In [3]:
started = time.perf_counter()
transformer_har_results = run_har_transformer(False)
print("Transformer HAR runtime seconds:", time.perf_counter() - started)
assert len(transformer_har_results) == 9


Transformer HAR runtime seconds: 188.42976020600003


## Full head-count ablation

Measures one, four, and eight heads at fixed width.

In [4]:
started = time.perf_counter()
head_results = run_transformer_ablations(False)
print("Head ablation runtime seconds:", time.perf_counter() - started)
assert len(head_results) == 3
assert len({row[2] for row in head_results}) == 1


Head ablation runtime seconds: 0.1426375110000322


## Full causal-mask and context-length study

Measures causal versus unmasked forward passes for contexts 64, 128, 256, and 512.

In [5]:
started = time.perf_counter()
context_results = run_mask_and_context_ablation(False)
print("Context ablation runtime seconds:", time.perf_counter() - started)
assert len(context_results) == 4


Context ablation runtime seconds: 0.04888825400001906


## Full Tiny Shakespeare language model

Trains the causal character model and writes the validation curve and metrics.

In [6]:
started = time.perf_counter()
model, chars, encode, lm_history = train_language_model(False)
print(lm_history[-1], lm_history[-1] / torch.log(torch.tensor(2.0)).item())
print("Language-model runtime seconds:", time.perf_counter() - started)
assert len(lm_history) == 12


1.6035678694811037 2.3134594066722634
Language-model runtime seconds: 109.30247081599998


## Required generated samples and attention heatmaps

Writes 500-character samples at temperatures 0.2, 0.8, and 1.5, then saves two attention heatmaps.

In [7]:
generate_samples(model, chars, encode, ROOT / 'problem2' / 'generated_samples.txt')
prompt = 'First Citizen:\nBefore we proceed any further, hear me speak.'
tokens = torch.tensor([[encode[ch] for ch in prompt]], device=next(model.parameters()).device)
plot_attention_heatmaps(model, tokens, ROOT / 'problem2' / 'figures' / 'p2_attention_heatmaps.png')
print('samples and heatmap saved')

samples and heatmap saved


In [8]:
import os
from google.colab import files

P2 = "/content/BITS_Assignments/1st_Semester/DNN/2nd Assignment/2024SAMPLE_DNN_Assignment/problem2"
os.chdir(P2)

!ls -l figures/p2_attention_heatmaps.png context_ablation.csv transformer_ablations.csv
!cd .. && zip -qr /content/problem2_outputs.zip problem2/figures problem2/*.csv problem2/generated_samples.txt

files.download("/content/problem2_outputs.zip")

-rw-r--r-- 1 root root   472 Nov 12 19:19 context_ablation.csv
-rw-r--r-- 1 root root 38467 Nov 12 19:21 figures/p2_attention_heatmaps.png
-rw-r--r-- 1 root root   110 Nov 12 19:19 transformer_ablations.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>